# piscan — prompt injection zaifliklarini topish (Qwen2.5-7B)

Bu notebook BJAI maqolasi uchun tajriba o'tkazadi.

**Qadamlar:**
1. Kaggle'da yangi Notebook oching.
2. O'ng paneldan **Accelerator → GPU T4 x2** (yoki P100) ni tanlang.
3. **Settings → Internet → ON** qilib qo'ying (model yuklab olinadi).
4. Ushbu notebook'ni to'liq ishga tushiring (Run All).
5. Oxirida chiqadigan `runs_qwen.jsonl` va `tables.json` fayllarini yuklab olib, menga qaytaring.

Butun ishlash ~30–60 daqiqa (model hajmi va payload soniga qarab).

## 1. Kodni olish

Ikki variant bor. **A variant**: `piscan` papkasini Kaggle Dataset sifatida yuklagan bo'lsangiz, u `/kaggle/input/...` da turadi. **B variant**: pastdagi katak kodni to'g'ridan-to'g'ri yozib qo'yadi (dataset shart emas).

In [ ]:
# piscan kodini Kaggle Dataset ichidan topib, ish papkasiga ko'chiradi.
# Oldin o'ng paneldan "+ Add Input" bosib, yuklagan piscan datasetini qo'shing.
import os, glob, shutil

found = glob.glob("/kaggle/input/**/run.py", recursive=True)
if found:
    src_dir = os.path.dirname(found[0])
    for item in os.listdir(src_dir):
        s = os.path.join(src_dir, item)
        d = os.path.join("/kaggle/working", item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy(s, d)
    os.chdir("/kaggle/working")
    print("kod ko'chirildi:", src_dir, "->", os.getcwd())
    print("fayllar:", sorted(os.listdir(".")))
else:
    print("run.py topilmadi. piscan datasetini '+ Add Input' orqali qo'shing.")

## 2. Kutubxonalar

In [ ]:
!pip -q install "transformers>=4.44" "accelerate>=0.30" 2>/dev/null
import torch
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yo'q")

## 3. Kodni tekshirish (mock, CPU)

Avval modelsiz, soxta backend bilan quvur ishlashini tekshiramiz.

In [ ]:
!python run.py --mock --out runs_mock.jsonl
!python analyze.py runs_mock.jsonl | head -25

## 4. Haqiqiy tajriba (Qwen2.5-7B-Instruct)

Agar vaqt kam bo'lsa, `--scripts` va `--templates` bilan kichraytiring.

In [ ]:
!python run.py --model Qwen/Qwen2.5-7B-Instruct --out runs_qwen.jsonl --verbose

## 5. Jadvallarni chiqarish

In [ ]:
!python analyze.py runs_qwen.jsonl

## 6. Natijani yuklab olish

Quyidagi ikki faylni Kaggle'ning o'ng panelidagi **Output** bo'limidan yuklab oling va menga qaytaring:
- `runs_qwen.jsonl`
- `tables.json`

In [ ]:
import shutil
for f in ["runs_qwen.jsonl", "tables.json"]:
    try:
        shutil.copy(f, "/kaggle/working/"+f)
        print("tayyor:", f)
    except Exception as e:
        print("xato:", f, e)